# Qwen2.5-3B Skill NER — Sayfullina Dataset
Fine-tune Qwen2.5-3B-Instruct with LoRA on the Sayfullina skill extraction dataset.
Target: T4 x2 Kaggle GPU, single-GPU training via CUDA_VISIBLE_DEVICES=0

In [1]:
# Cell 1 — Install
import subprocess
subprocess.run(["pip", "install", "-q",
    "transformers==4.46.3",
    "trl==0.12.2",
    "peft==0.13.2",
    "accelerate",
    "datasets",
    "seqeval",
    "tqdm"
])
print("Done. Restart kernel now, then run all cells below.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.8 MB/s eta 0:00:00
Done. Restart kernel now, then run all cells below.


In [2]:
# Cell 2 — Imports
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # hide second GPU — prevents multi-device split
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import random
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset, Dataset
from seqeval.metrics import f1_score, classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

random.seed(42)
torch.manual_seed(42)
print("Imports OK")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Visible GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

2026-05-15 15:46:01.438871: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778859961.657725      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778859961.726314      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778859962.286463      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778859962.286502      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778859962.286505      23 computation_placer.cc:177] computation placer alr

Imports OK
PyTorch: 2.10.0+cu128
CUDA available: True
Visible GPUs: 1
GPU: Tesla T4
Compute capability: (7, 5)


In [3]:
# Cell 3 — Load Sayfullina dataset
ds = load_dataset("jjzha/sayfullina")
print("Train:", len(ds["train"]))
print("Test: ", len(ds["test"]))
print("Sample:", ds["train"][0])

README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3705 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1855 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1851 [00:00<?, ? examples/s]

Train: 3705
Test:  1851
Sample: {'idx': 0, 'tokens': ['excellent', 'organisational', 'skill', 'and', 'able', 'to', 'multitask'], 'tags_skill': ['O', 'O', 'O', 'O', 'B-SOFT', 'I-SOFT', 'I-SOFT']}


In [4]:
# Cell 4 — BIO helpers
def get_tokens_and_tags(example):
    return example["tokens"], example["tags_skill"]


def to_bracket_format(tokens, tags):
    result = []
    i = 0
    while i < len(tokens):
        if tags[i].startswith("B-"):
            span = [tokens[i]]
            j = i + 1
            while j < len(tokens) and tags[j].startswith("I-"):
                span.append(tokens[j])
                j += 1
            result.append(f"@@{' '.join(span)}##")
            i = j
        else:
            result.append(tokens[i])
            i += 1
    return " ".join(result)


def bio_from_bracket(tokens, output_text):
    tags = ["O"] * len(tokens)
    spans = []
    text = output_text
    while "@@" in text and "##" in text:
        start = text.find("@@")
        end   = text.find("##", start)
        if start == -1 or end == -1:
            break
        span = text[start+2:end].strip()
        spans.append(span)
        text = text[end+2:]
    for span in spans:
        span_tokens = span.split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i : i + n] == span_tokens:
                if all(tags[i + j] == "O" for j in range(n)):
                    tags[i] = "B-SOFT"
                    for j in range(1, n):
                        tags[i + j] = "I-SOFT"
                break
    return tags

print("BIO helpers defined")

BIO helpers defined


In [5]:
# Cell 5 — Build training set
SYSTEM = (
    "You are a skill-span extractor for job advertisements. "
    "Given a sentence, wrap every skill span with @@...## brackets. "
    "Copy the sentence exactly, only adding @@ before and ## after each skill. "
    "If there are no skills, return the sentence unchanged."
)

def format_prompt(sentence, target=None):
    prompt = (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{sentence}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    if target is not None:
        prompt += f"{target}<|im_end|>"
    return prompt


def build_training_set(dataset):
    positives, negatives = [], []
    for ex in dataset["train"]:
        tokens, tags = get_tokens_and_tags(ex)
        sentence = " ".join(tokens)
        target   = to_bracket_format(tokens, tags)
        entry    = {"text": format_prompt(sentence, target)}
        if "@@" in target:
            positives.append(entry)
        else:
            negatives.append(entry)

    max_neg   = int(len(positives) * 0.3)
    negatives = random.sample(negatives, min(max_neg, len(negatives)))
    combined  = positives + negatives
    random.shuffle(combined)
    print(f"Positives: {len(positives)}, Hard negatives: {len(negatives)}, Total: {len(combined)}")
    return combined


train_data = build_training_set(ds)
print("Sample prompt:")
print(train_data[0]["text"][:300])

Positives: 3701, Hard negatives: 4, Total: 3705
Sample prompt:
<|im_start|>system
You are a skill-span extractor for job advertisements. Given a sentence, wrap every skill span with @@...## brackets. Copy the sentence exactly, only adding @@ before and ## after each skill. If there are no skills, return the sentence unchanged.<|im_end|>
<|im_start|>user
and dri


In [6]:
# Cell 6 — Prepare HuggingFace dataset
hf_train = Dataset.from_list(train_data)
print(f"Training examples: {len(hf_train)}")

Training examples: 3705


In [7]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")  # should be ~15GB


Free GPU: 15.5 GB


In [8]:
# Cell 7 — Load model
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"":"cuda:0"},
    trust_remote_code=True,
)
model.config.use_cache = False
print(f"Loaded on: {next(model.parameters()).device}")
print(f"Visible GPUs: {torch.cuda.device_count()}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded on: cuda:0
Visible GPUs: 1
GPU memory used: 6.2 GB


In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)  # let it cast adapters to fp32 normally
model.print_trainable_parameters()
model.enable_input_require_grads()
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB")


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
GPU memory after LoRA: 6.3 GB


In [10]:
sft_config = SFTConfig(
    output_dir                    = "/kaggle/working/checkpoints",
    per_device_train_batch_size   = 2,
    gradient_accumulation_steps   = 8,
    num_train_epochs              = 3,
    learning_rate                 = 2e-4,
    warmup_ratio                  = 0.1,
    lr_scheduler_type             = "cosine",
    fp16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    logging_steps                 = 20,
    save_strategy                 = "epoch",
    report_to                     = "none",
    seed                          = 42,
    optim                         = "adamw_torch",
    ddp_find_unused_parameters    = False,
    dataloader_pin_memory         = False,
    dataset_text_field            = "text",
    max_seq_length                = 384,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = hf_train,
    args          = sft_config,
)

trainer.train()
print("Training complete.")


Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Step,Training Loss
20,3.287500
40,0.849300
60,0.693200
80,0.638700
100,0.627000
120,0.615600
140,0.617500
160,0.609500
180,0.595600
200,0.592500


Training complete.


In [11]:
# Cell 10 — Save adapter
model.save_pretrained("/kaggle/working/lora_adapter")
tokenizer.save_pretrained("/kaggle/working/lora_adapter")

print("Saved files:", os.listdir("/kaggle/working/lora_adapter"))

Saved files: ['tokenizer_config.json', 'vocab.json', 'added_tokens.json', 'adapter_model.safetensors', 'tokenizer.json', 'README.md', 'special_tokens_map.json', 'merges.txt', 'adapter_config.json']


In [12]:
# Cell 11 — Evaluate
model.eval()

test_examples = list(ds["test"])
random.shuffle(test_examples)
test_examples = test_examples[:200]

true_tags = []
pred_tags = []

for example in tqdm(test_examples, desc="Evaluating"):
    tokens, gold = get_tokens_and_tags(example)
    prompt       = format_prompt(" ".join(tokens))

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=384,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 128,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    pred = bio_from_bracket(tokens, decoded)

    true_tags.append(gold)
    pred_tags.append(pred)

print("\nDone.")

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Evaluating: 100%|██████████| 200/200 [06:12<00:00,


Done.


In [13]:
# Cell 12 — Results
print("=== Sayfullina — Qwen2.5-3B SFT (Phase 4) ===")
print(f"Examples evaluated: {len(true_tags)}")
print()
print(classification_report(true_tags, pred_tags))
f1 = f1_score(true_tags, pred_tags)
print(f"Strict F1: {f1:.4f}")
print()
print("Phase 1 (random 5-shot, no RAG):        0.1432 F1")
print("Phase 2 (RAG-1 + RAG-2, no verifier):   0.3154 F1")
print("Phase 3 (+ Verifier):                   0.3429 F1")
print("Paper SRICL target (full pipeline):      0.6118 F1")

=== Sayfullina — Qwen2.5-3B SFT (Phase 4) ===
Examples evaluated: 200

              precision    recall  f1-score   support

        SOFT       0.81      0.81      0.81       201

   micro avg       0.81      0.81      0.81       201
   macro avg       0.81      0.81      0.81       201
weighted avg       0.81      0.81      0.81       201

Strict F1: 0.8100

Phase 1 (random 5-shot, no RAG):        0.1432 F1
Phase 2 (RAG-1 + RAG-2, no verifier):   0.3154 F1
Phase 3 (+ Verifier):                   0.3429 F1
Paper SRICL target (full pipeline):      0.6118 F1


In [14]:
# Cell 13 — Download reminder
print("Adapter files:", os.listdir("/kaggle/working/lora_adapter"))
print("\nDOWNLOAD THE ADAPTER BEFORE CLOSING THE SESSION")
print("Kaggle sidebar → Output → lora_adapter/ → Download")

Adapter files: ['tokenizer_config.json', 'vocab.json', 'added_tokens.json', 'adapter_model.safetensors', 'tokenizer.json', 'README.md', 'special_tokens_map.json', 'merges.txt', 'adapter_config.json']

DOWNLOAD THE ADAPTER BEFORE CLOSING THE SESSION
Kaggle sidebar → Output → lora_adapter/ → Download
